[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/B.%20Warehouse%20%26%20Fulfillment%20Center%20Operations%20%28Inside%20the%20Four%20Walls%29/059.%20The%20Warehouse%20Layout%20Design%20Problem/P59-Tier-7_executed.ipynb)

---


# 59. The Warehouse Layout Design Problem

## Tier 7: Human–AI Partnership

### Purpose and scope

This notebook illustrates a human–AI decision process for a warehouse layout problem. AI-generated candidate layouts are evaluated with a flow-distance measure, reviewed using explicit rule-based expert rubrics, ranked by a weighted consensus score, and considered for iterative refinement. The process is a teaching model of collaboration; it is not a validated deployment study or a substitute for input from actual warehouse experts.

The inputs are a set of warehouse locations, a current layout, and directed material flows. The principal quantitative output is each candidate's flow-weighted Manhattan distance and its percentage change relative to the current layout. A pick-time reduction is reported only as a proxy derived from flow-distance improvement. The notebook does not contain measurements needed to establish space utilization, safety, automation readiness, implementation cost, implementation duration, staff satisfaction, or realized economic impact. Those objectives are therefore identified as unquantified rather than presented as measured results.

### Model assumptions

- Locations are points on a two-dimensional grid, and travel distance between two locations is Manhattan distance: horizontal distance plus vertical distance.
- Each flow value is a nonnegative relative intensity. The model does not assign physical units to it, so total flow-weighted distance is a relative score, not a distance in feet or meters.
- The flow list is directed: a flow from area \(i\) to area \(j\) is counted as supplied. The notebook does not infer a reverse flow.
- Each candidate assigns each named area to one location. This example does not impose exclusivity, aisle, capacity, safety, or adjacency constraints.
- The relationship between flow-distance improvement and pick-time reduction is a simple proxy assumption, not a calibrated operational law.
- Expert reviews are deterministic teaching rubrics based on the available model outputs. They are not observations or decisions made by real people.

### Partnership process

The sequence is: define objectives; generate candidate layouts; calculate and explain model-based performance; obtain role-specific rule-based reviews; rank candidates using the stated human–AI weighting; refine the top candidates using review feedback; and select a final candidate. This retains the collaborative decision sequence while distinguishing calculated quantities from information that the input data cannot support.

### Mathematical model

Let \(A\) be the set of warehouse areas, \(L\) the set of grid locations, and \(F\) the set of directed area pairs with flow intensity \(f_{ij}\). A layout \(x\) assigns a location \(x_i \in L\) to each area \(i \in A\). For grid coordinates \((u_l,v_l)\), the distance between locations \(l\) and \(m\) is

$$
d(l,m)=|u_l-u_m|+|v_l-v_m|.
$$

The flow-weighted distance of layout \(x\) is

$$
D(x)=\sum_{(i,j)\in F} f_{ij}\,d(x_i,x_j).
$$

For the current layout \(x^0\), the percentage improvement of candidate \(x\) is

$$
I(x)=100\,\frac{D(x^0)-D(x)}{D(x^0)},
$$
provided \(D(x^0)>0\). Positive values indicate a lower flow-weighted distance than the current layout; negative values indicate a higher one. The pick-time reduction proxy is \(P(x)=0.8I(x)\), following the conversion assumption encoded in the example. This proxy is used only as an illustrative comparison with the stated pick-time target.

The AI alignment score uses the available pick-time proxy only. If the target is \(T>0\), its achievement rate is

$$
a(x)=\min\!\left(1,\max\!\left(0,\frac{P(x)}{T}\right)\right),
$$
so the score is bounded between 0 and 1. The consensus score retains the notebook's human–AI weighting:

$$
C(x)=0.4\,[10a(x)]+0.6\,H(x),
$$
where \(H(x)\) is the weighted mean of the rubric ratings on a 0–10 scale. The coefficients are teaching-model inputs, not empirically estimated preferences. Objectives without supporting data are not included in \(a(x)\).

In the code, `locations`, `flows`, and a layout dictionary represent the sets, flow values, and assignment \(x\). ` _calculate_total_flow_distance` computes \(D(x)\), ` _ai_analyze_alternative` calculates the proxy and alignment score, and `collaborative_decision_making` computes \(C(x)\). The candidate set is finite and explicitly listed; this notebook ranks those candidates rather than claiming to solve a general warehouse-layout optimization problem. For \(n\) flow pairs, evaluating one layout takes \(O(n)\) time; evaluating \(k\) candidates takes \(O(kn)\), apart from sorting their scores.

### Interpreting results

The execution-generated rankings compare only the listed candidates under the stated distance measure, proxy, review rubrics, and weights. They are not evidence that a layout achieves real-world operational targets. A practical decision would require validated operational, safety, cost, and implementation data, as well as review by qualified staff.

In [ ]:
from itertools import product
import random

import matplotlib.pyplot as plt
import numpy as np

# Keep the example reproducible if a later teaching exercise uses randomness.
np.random.seed(42)
random.seed(42)

plt.style.use("default")

In [ ]:
class HumanAIPartnership:
    """Coordinate a rule-based human–AI review of candidate warehouse layouts."""

    def __init__(self, warehouse_config):
        self.config = warehouse_config
        self.human_experts = self._initialize_human_experts()
        self.ai_agents = self._initialize_ai_agents()
        self.partnership_history = []
        self.current_iteration = 0
        self.consensus_solutions = []

        required_areas = set(self.config["current_layout"])
        if set(self.config["locations"]) == set():
            raise ValueError("At least one warehouse location is required.")
        if not required_areas.issubset(self.config["current_layout"]):
            raise ValueError("The current layout must assign every required area.")

    def _initialize_human_experts(self):
        """Define review roles and their contribution to the consensus score."""
        return {
            "warehouse_manager": {
                "name": "Warehouse Manager",
                "role": "Operational feasibility",
                "decision_weight": 0.25,
            },
            "industrial_engineer": {
                "name": "Industrial Engineer",
                "role": "Flow and layout performance",
                "decision_weight": 0.25,
            },
            "safety_coordinator": {
                "name": "Safety Coordinator",
                "role": "Safety evidence and constraints",
                "decision_weight": 0.20,
            },
            "it_specialist": {
                "name": "IT Specialist",
                "role": "Automation and infrastructure evidence",
                "decision_weight": 0.15,
            },
            "operations_analyst": {
                "name": "Operations Analyst",
                "role": "Performance and cost evidence",
                "decision_weight": 0.15,
            },
        }

    def _initialize_ai_agents(self):
        """Describe the specialized analytical roles represented in this example."""
        return {
            "flow_optimization_agent": {
                "name": "Flow Analysis Agent",
                "specialization": "flow-weighted distance calculation",
            },
            "space_planning_agent": {
                "name": "Space Planning Agent",
                "specialization": "space-related objective identification",
            },
            "risk_assessment_agent": {
                "name": "Risk Review Agent",
                "specialization": "identification of unquantified risk evidence",
            },
        }

    def define_strategic_objectives(self):
        """State the targets and mark which objectives the supplied data can assess."""
        objectives = {
            "pick_time_reduction": {
                "target": 25,
                "unit": "%",
                "priority": "high",
                "assessable": True,
                "measure": "Flow-distance-based proxy; not observed pick time",
            },
            "space_utilization_improvement": {
                "target": 15,
                "unit": "%",
                "priority": "high",
                "assessable": False,
                "measure": "No capacity or area-use measurements supplied",
            },
            "safety_standards_maintenance": {
                "target": 0,
                "unit": "incidents",
                "priority": "critical",
                "assessable": False,
                "measure": "No safety or incident data supplied",
            },
            "automation_readiness": {
                "target": 80,
                "unit": "% readiness score",
                "priority": "medium",
                "assessable": False,
                "measure": "No infrastructure or readiness assessment supplied",
            },
            "cost_effectiveness": {
                "target": 500000,
                "unit": "maximum cost in dollars",
                "priority": "medium",
                "assessable": False,
                "measure": "No candidate-specific cost data supplied",
            },
            "implementation_timeline": {
                "target": 4,
                "unit": "weeks maximum",
                "priority": "medium",
                "assessable": False,
                "measure": "No candidate-specific schedule data supplied",
            },
        }
        print("Strategic objectives and data coverage:")
        for name, objective in objectives.items():
            coverage = "assessable as a proxy" if objective["assessable"] else "not assessable"
            print(
                f"  {name.replace('_', ' ').title()}: target {objective['target']} "
                f"{objective['unit']} ({coverage})"
            )
        return objectives

    def generate_ai_alternatives(self, strategic_objectives):
        """Create the six explicitly specified candidate layouts."""
        approaches = [
            "flow_optimization_focused",
            "space_efficiency_focused",
            "balanced_optimization",
            "automation_ready",
            "cost_minimized",
            "risk_averse",
        ]
        alternatives = {}
        for index, approach in enumerate(approaches, start=1):
            layout = self._generate_layout_by_approach(approach)
            analysis = self._ai_analyze_alternative(
                layout, approach, strategic_objectives
            )
            alternatives[f"alternative_{index}"] = {
                "layout": layout,
                "approach": approach,
                "ai_analysis": analysis,
                "confidence_score": analysis["overall_confidence"],
            }
        print(f"Generated {len(alternatives)} explicitly defined candidate layouts.")
        return alternatives

    def _generate_layout_by_approach(self, approach):
        """Return a candidate assignment using the example's location indices."""
        layouts = {
            "flow_optimization_focused": {
                "Receiving": 19, "Returns": 18, "Fast_Storage": 14,
                "Medium_Storage": 9, "Slow_Storage": 4, "Packing": 13,
                "Shipping": 8,
            },
            "space_efficiency_focused": {
                "Receiving": 19, "Returns": 0, "Fast_Storage": 14,
                "Medium_Storage": 9, "Slow_Storage": 4, "Packing": 13,
                "Shipping": 8,
            },
            "balanced_optimization": {
                "Receiving": 19, "Returns": 18, "Fast_Storage": 13,
                "Medium_Storage": 9, "Slow_Storage": 4, "Packing": 14,
                "Shipping": 8,
            },
            "automation_ready": {
                "Receiving": 19, "Returns": 18, "Fast_Storage": 13,
                "Medium_Storage": 8, "Slow_Storage": 3, "Packing": 14,
                "Shipping": 9,
            },
            "cost_minimized": {
                "Receiving": 19, "Returns": 18, "Fast_Storage": 14,
                "Medium_Storage": 9, "Slow_Storage": 4, "Packing": 13,
                "Shipping": 8,
            },
            "risk_averse": self.config["current_layout"].copy(),
        }
        if approach not in layouts:
            raise ValueError(f"Unknown candidate approach: {approach}")
        layout = layouts[approach]
        self._validate_layout(layout)
        return layout

    def _validate_layout(self, layout):
        """Check that a candidate assigns every area to a known location."""
        if set(layout) != set(self.config["current_layout"]):
            raise ValueError("A candidate must assign every configured area exactly once.")
        location_count = len(self.config["locations"])
        if any(not isinstance(index, int) or not 0 <= index < location_count
               for index in layout.values()):
            raise ValueError("Every assigned location index must be valid.")

    def _ai_analyze_alternative(self, layout, approach, strategic_objectives):
        """Calculate the flow proxy and explicitly leave unsupported metrics unknown."""
        performance = self._calculate_layout_performance(layout)
        assessment = {}
        for name, objective in strategic_objectives.items():
            if name == "pick_time_reduction":
                achieved = performance["pick_time_reduction_proxy"]
                target = objective["target"]
                achievement_rate = min(1.0, max(0.0, achieved / target))
            else:
                achieved = None
                achievement_rate = None
            assessment[name] = {
                "target": objective["target"],
                "unit": objective["unit"],
                "achieved": achieved,
                "achievement_rate": achievement_rate,
                "assessable": objective["assessable"],
                "priority": objective["priority"],
                "measure": objective["measure"],
            }

        # Only the one quantitatively assessable objective contributes to this score.
        overall_confidence = assessment["pick_time_reduction"]["achievement_rate"]
        reasoning = self._generate_xai_reasoning(
            layout, approach, assessment, performance
        )
        return {
            "performance": performance,
            "objective_assessment": assessment,
            "overall_confidence": overall_confidence,
            "xai_reasoning": reasoning,
            "recommendation": self._generate_recommendation(overall_confidence),
        }

    def _calculate_layout_performance(self, layout):
        """Compute distance and a clearly identified pick-time proxy."""
        self._validate_layout(layout)
        candidate_distance = self._calculate_total_flow_distance(layout)
        baseline_distance = self._calculate_total_flow_distance(
            self.config["current_layout"]
        )
        if baseline_distance <= 0:
            raise ValueError("The current layout must have positive flow-weighted distance.")
        flow_improvement = 100 * (baseline_distance - candidate_distance) / baseline_distance
        return {
            "total_flow_distance": candidate_distance,
            "baseline_flow_distance": baseline_distance,
            "flow_improvement_pct": flow_improvement,
            "pick_time_reduction_proxy": 0.8 * flow_improvement,
            "space_utilization_improvement": None,
            "safety_score": None,
            "automation_readiness": None,
            "implementation_cost": None,
            "implementation_weeks": None,
        }

    def _calculate_total_flow_distance(self, layout):
        """Evaluate the directed sum of flow intensity times Manhattan distance."""
        locations = self.config["locations"]
        total_distance = 0.0
        for (area_from, area_to), flow_intensity in self.config["flows"].items():
            if flow_intensity < 0:
                raise ValueError("Flow intensities must be nonnegative.")
            if area_from not in layout or area_to not in layout:
                raise ValueError("Every area in the flow data must be in the layout.")
            x1, y1 = locations[layout[area_from]]
            x2, y2 = locations[layout[area_to]]
            distance = abs(x1 - x2) + abs(y1 - y2)
            total_distance += flow_intensity * distance
        return total_distance

    def _generate_xai_reasoning(self, layout, approach, assessment, performance):
        """Explain the calculated comparison and its data limitations."""
        return {
            "approach": approach.replace("_", " "),
            "flow_distance": performance["total_flow_distance"],
            "baseline_flow_distance": performance["baseline_flow_distance"],
            "flow_improvement_pct": performance["flow_improvement_pct"],
            "pick_time_reduction_proxy": performance["pick_time_reduction_proxy"],
            "unassessed_objectives": [
                name for name, item in assessment.items() if not item["assessable"]
            ],
            "interpretation": (
                "The pick-time figure is a proxy derived from flow distance; "
                "it is not an observed operational result."
            ),
        }

    def _generate_recommendation(self, confidence):
        """Describe alignment with the assessable target, not overall feasibility."""
        if confidence >= 0.8:
            return "Strong alignment with the assessable pick-time target; other objectives remain unassessed."
        if confidence >= 0.6:
            return "Partial alignment with the assessable pick-time target; other objectives remain unassessed."
        return "Limited alignment with the assessable pick-time target; other objectives remain unassessed."

    def human_expert_review(self, alternatives):
        """Apply transparent, role-specific teaching rubrics to each candidate."""
        reviews = {}
        for expert_id, expert in self.human_experts.items():
            reviews[expert_id] = {}
            for alt_id, alternative in alternatives.items():
                reviews[expert_id][alt_id] = self._expert_review_alternative(
                    expert_id, expert, alternative, alt_id
                )
        return reviews

    def _expert_review_alternative(self, expert_id, expert, alternative, alt_id):
        """Return a 0–10 rubric rating and note evidence gaps honestly."""
        performance = alternative["ai_analysis"]["performance"]
        flow_gain = performance["flow_improvement_pct"]
        pick_proxy = performance["pick_time_reduction_proxy"]
        rating = 5.0
        comments = []
        concerns = []
        suggestions = []

        if expert_id == "warehouse_manager":
            if pick_proxy >= 20:
                rating += 3
                comments.append("The modeled pick-time proxy is at least 20%.")
            else:
                concerns.append("The modeled pick-time proxy is below 20%.")
            comments.append("Operational disruption has not been measured.")
        elif expert_id == "industrial_engineer":
            if flow_gain >= 25:
                rating += 3
                comments.append("Flow-weighted distance improves by at least 25%.")
            elif flow_gain > 0:
                rating += 1
                comments.append("Flow-weighted distance improves relative to the baseline.")
            else:
                concerns.append("Flow-weighted distance does not improve relative to the baseline.")
            comments.append("The assessment uses the specified grid-distance model.")
        elif expert_id == "safety_coordinator":
            concerns.append("Safety cannot be assessed from the supplied layout and flow data.")
            suggestions.append("Obtain a qualified safety review before implementation.")
        elif expert_id == "it_specialist":
            concerns.append("Infrastructure and automation readiness data are not supplied.")
            suggestions.append("Assess infrastructure requirements before implementation.")
        elif expert_id == "operations_analyst":
            concerns.append("Candidate-specific cost and schedule data are not supplied.")
            suggestions.append("Estimate implementation cost and duration before selection.")

        rating = min(10.0, max(0.0, rating))
        if rating >= 8 and not concerns:
            feasibility = "high within the assessed criteria"
        elif rating >= 6:
            feasibility = "partially assessed"
        else:
            feasibility = "insufficient evidence"
        return {
            "alternative_id": alt_id,
            "expert_rating": rating,
            "expert_comments": comments,
            "concerns": concerns,
            "suggestions": suggestions,
            "implementation_feasibility": feasibility,
            "review_role": expert["role"],
        }

    def collaborative_decision_making(
        self, alternatives, expert_reviews, strategic_objectives
    ):
        """Rank candidates using the stated 40% AI and 60% human score weights."""
        del strategic_objectives  # The assessment is already stored with each candidate.
        consensus_scores = {}
        for alt_id, alternative in alternatives.items():
            ai_score = 10 * alternative["ai_analysis"]["overall_confidence"]
            scores = []
            weights = []
            for expert_id, expert in self.human_experts.items():
                review = expert_reviews[expert_id][alt_id]
                scores.append(review["expert_rating"])
                weights.append(expert["decision_weight"])
            human_consensus = float(np.average(scores, weights=weights))
            consensus_score = 0.4 * ai_score + 0.6 * human_consensus
            consensus_scores[alt_id] = {
                "consensus_score": consensus_score,
                "ai_score": ai_score,
                "human_consensus": human_consensus,
                "alternative": alternative,
            }
        ranked = sorted(
            consensus_scores.items(),
            key=lambda item: (-item[1]["consensus_score"], item[0]),
        )
        self.consensus_solutions = ranked
        print("Ranked candidates by the defined consensus score:")
        for rank, (alt_id, scores) in enumerate(ranked, start=1):
            print(
                f"  {rank}. {alt_id}: {scores['consensus_score']:.2f}/10 "
                f"(AI alignment: {scores['ai_score']:.2f}/10; "
                f"rubric consensus: {scores['human_consensus']:.2f}/10)"
            )
        return ranked

    def iterative_refinement(
        self, top_alternatives, expert_reviews, strategic_objectives
    ):
        """Reassess the top three candidates after recording reviewer feedback."""
        refined = {}
        for alt_id, scores in top_alternatives[:3]:
            alternative = scores["alternative"]
            concerns = []
            suggestions = []
            for expert_id in self.human_experts:
                review = expert_reviews[expert_id][alt_id]
                concerns.extend(review["concerns"])
                suggestions.extend(review["suggestions"])

            refined_layout = self._refine_layout_based_on_feedback(
                alternative["layout"], concerns, suggestions
            )
            analysis = self._ai_analyze_alternative(
                refined_layout, alternative["approach"], strategic_objectives
            )
            refined_id = f"{alt_id}_refined"
            refined[refined_id] = {
                "layout": refined_layout,
                "approach": alternative["approach"],
                "ai_analysis": analysis,
                "original_alternative": alt_id,
                "concerns_addressed": concerns,
                "suggestions_incorporated": suggestions,
            }
            print(
                f"Reviewed {refined_id}: {len(concerns)} evidence or feasibility "
                f"concerns recorded; no unsupported layout changes were inferred."
            )
        self.current_iteration += 1
        return refined

    def _refine_layout_based_on_feedback(self, original_layout, concerns, suggestions):
        """Retain the candidate assignment when feedback lacks a quantified constraint."""
        del concerns, suggestions
        return original_layout.copy()

    def final_selection(self, refined_alternatives, expert_reviews):
        """Score refined candidates with the same weights and role-specific rubrics."""
        final_scores = {}
        for alt_id, alternative in refined_alternatives.items():
            ai_score = 10 * alternative["ai_analysis"]["overall_confidence"]
            human_scores = []
            weights = []
            for expert_id, expert in self.human_experts.items():
                review = self._expert_review_alternative(
                    expert_id, expert, alternative, alt_id
                )
                # Feedback is recorded for transparency; it does not imply a verified fix.
                review["concerns_addressed"] = alternative["concerns_addressed"]
                human_scores.append(review["expert_rating"])
                weights.append(expert["decision_weight"])
            human_consensus = float(np.average(human_scores, weights=weights))
            score = 0.4 * ai_score + 0.6 * human_consensus
            final_scores[alt_id] = {
                "final_score": score,
                "ai_score": ai_score,
                "human_consensus": human_consensus,
                "alternative": alternative,
            }

        best_id = sorted(
            final_scores,
            key=lambda name: (-final_scores[name]["final_score"], name),
        )[0]
        best = final_scores[best_id]
        tied = [
            name for name, item in final_scores.items()
            if np.isclose(item["final_score"], best["final_score"], rtol=1e-9, atol=1e-12)
        ]
        print(f"Selected candidate by the stated rubric: {best_id}")
        print(f"Consensus score: {best['final_score']:.2f}/10")
        if len(tied) > 1:
            print(f"Candidates tied within numerical tolerance: {', '.join(sorted(tied))}")
        print("This ranking does not establish implementation feasibility.")
        return best_id, best

    def generate_partnership_report(self, selected_solution, implementation_results=None):
        """Report model outputs and distinguish them from implementation evidence."""
        alternative = selected_solution["alternative"]
        performance = alternative["ai_analysis"]["performance"]
        print("\nHUMAN–AI PARTNERSHIP REPORT")
        print(f"  Human review roles: {len(self.human_experts)}")
        print(f"  Analytical roles represented: {len(self.ai_agents)}")
        print(f"  Explicit candidate layouts: 6")
        print(f"  Refinement rounds: {self.current_iteration}")
        print(f"  Selected candidate: {alternative.get('original_alternative', 'not recorded')}")
        print(f"  Flow-weighted distance: {performance['total_flow_distance']:.2f}")
        print(f"  Baseline flow-weighted distance: {performance['baseline_flow_distance']:.2f}")
        print(f"  Flow-distance improvement: {performance['flow_improvement_pct']:.2f}%")
        print(
            "  Pick-time reduction proxy: "
            f"{performance['pick_time_reduction_proxy']:.2f}% (not measured)"
        )
        print("  Space, safety, cost, readiness, and schedule: not quantified")
        if implementation_results:
            print("  Supplied implementation observations:")
            for metric, value in implementation_results.items():
                print(f"    {metric}: {value}")
        else:
            print("  Implementation observations: none supplied")
        print("  Interpretation: candidate ranking under the stated model and rubrics only.")

In [ ]:
warehouse_config = {
    "warehouse_size": 200000,  # Square feet; not used to infer capacity improvement.
    "daily_orders": 15000,  # Orders per day; not used to infer observed pick time.
    "locations": [
        (1, 4), (2, 4), (3, 4), (4, 4), (5, 4),
        (1, 3), (2, 3), (3, 3), (4, 3), (5, 3),
        (1, 2), (2, 2), (3, 2), (4, 2), (5, 2),
        (1, 1), (2, 1), (3, 1), (4, 1), (5, 1),
    ],
    "current_layout": {
        "Receiving": 19,
        "Returns": 18,
        "Fast_Storage": 14,
        "Medium_Storage": 9,
        "Slow_Storage": 4,
        "Packing": 13,
        "Shipping": 8,
    },
    "flows": {
        ("Receiving", "Returns"): 50,
        ("Receiving", "Fast_Storage"): 480,
        ("Receiving", "Medium_Storage"): 280,
        ("Receiving", "Slow_Storage"): 120,
        ("Returns", "Fast_Storage"): 30,
        ("Fast_Storage", "Packing"): 450,
        ("Medium_Storage", "Packing"): 250,
        ("Slow_Storage", "Packing"): 100,
        ("Packing", "Shipping"): 800,
    },
}

assert set(warehouse_config["current_layout"]) == {
    area for flow in warehouse_config["flows"] for area in flow
}
assert len(warehouse_config["locations"]) == 20
print("Warehouse configuration loaded.")
print(f"  Grid locations: {len(warehouse_config['locations'])}")
print(f"  Named areas: {len(warehouse_config['current_layout'])}")
print(f"  Directed flow pairs: {len(warehouse_config['flows'])}")

## Run the collaborative decision process

The next cell follows the method's sequence: define objectives, generate candidates, review them, calculate consensus rankings, refine the leading candidates, and make a final rubric-based selection. The ratings are generated by the transparent rules in the class; they are not human-subject data. Unsupported objectives remain visible as evidence gaps throughout the process.

In [ ]:
partnership = HumanAIPartnership(warehouse_config)

strategic_objectives = partnership.define_strategic_objectives()
ai_alternatives = partnership.generate_ai_alternatives(strategic_objectives)
expert_reviews = partnership.human_expert_review(ai_alternatives)
ranked_alternatives = partnership.collaborative_decision_making(
    ai_alternatives, expert_reviews, strategic_objectives
)
refined_alternatives = partnership.iterative_refinement(
    ranked_alternatives, expert_reviews, strategic_objectives
)
selected_solution_id, selected_solution = partnership.final_selection(
    refined_alternatives, expert_reviews
)

# Verify that every candidate's reported distance is computed from its layout.
for candidate in ai_alternatives.values():
    expected = partnership._calculate_total_flow_distance(candidate["layout"])
    reported = candidate["ai_analysis"]["performance"]["total_flow_distance"]
    assert np.isclose(expected, reported)

print("Candidate evaluation and selection completed.")

## Implementation evidence and report

A modeled candidate score is not an implementation result. No observed post-implementation measurements are supplied in this example, so the report deliberately receives no implementation-results dictionary. If a real evaluation later supplies observations, they should be recorded with units, measurement dates, baseline definitions, and data sources; they should not be inferred from the model's proxy.

In [ ]:
implementation_results = None
partnership.generate_partnership_report(
    selected_solution,
    implementation_results=implementation_results,
)

In [ ]:
def visualize_human_ai_partnership(partnership, ranked_alternatives, selected_solution):
    """Plot candidate model scores and the stated consensus components."""
    candidate_ids = [name for name, _ in ranked_alternatives]
    flow_distances = [
        scores["alternative"]["ai_analysis"]["performance"]["total_flow_distance"]
        for _, scores in ranked_alternatives
    ]
    consensus_scores = [scores["consensus_score"] for _, scores in ranked_alternatives]
    ai_scores = [scores["ai_score"] for _, scores in ranked_alternatives]
    human_scores = [scores["human_consensus"] for _, scores in ranked_alternatives]

    fig, axes = plt.subplots(1, 3, figsize=(17, 5))
    axes[0].bar(candidate_ids, flow_distances, color="steelblue", label="Candidate")
    baseline = partnership._calculate_total_flow_distance(
        partnership.config["current_layout"]
    )
    axes[0].axhline(
        baseline, color="darkorange", linestyle="--", label="Current layout"
    )
    axes[0].set_title("Flow-weighted distance")
    axes[0].set_ylabel("Relative flow-distance units")
    axes[0].set_xlabel("Candidate layout")
    axes[0].tick_params(axis="x", rotation=45)
    axes[0].legend()
    axes[0].grid(axis="y", alpha=0.3)

    axes[1].bar(candidate_ids, consensus_scores, color="seagreen")
    axes[1].set_title("Human–AI consensus ranking")
    axes[1].set_ylabel("Consensus score (0–10)")
    axes[1].set_xlabel("Candidate layout")
    axes[1].tick_params(axis="x", rotation=45)
    axes[1].grid(axis="y", alpha=0.3)

    x = np.arange(len(candidate_ids))
    width = 0.36
    axes[2].bar(x - width / 2, ai_scores, width, label="AI alignment (40%)")
    axes[2].bar(x + width / 2, human_scores, width, label="Rubric consensus (60%)")
    axes[2].set_title("Consensus score components")
    axes[2].set_ylabel("Score (0–10)")
    axes[2].set_xlabel("Candidate layout")
    axes[2].set_xticks(x)
    axes[2].set_xticklabels(candidate_ids, rotation=45, ha="right")
    axes[2].legend()
    axes[2].grid(axis="y", alpha=0.3)

    selected_name = selected_solution["alternative"].get(
        "original_alternative", "Selected candidate"
    )
    fig.suptitle(
        f"Candidate comparison; selected by the rubric: {selected_name}",
        fontsize=14,
    )
    fig.tight_layout()
    plt.show()


visualize_human_ai_partnership(
    partnership, ranked_alternatives, selected_solution
)

## Economic interpretation and limits

Economic impact requires measured operational changes and defensible monetary assumptions. The supplied warehouse size and order volume alone do not establish space value, time savings, labor savings, or return on investment. In particular, a flow-distance proxy cannot be treated as an observed reduction in pick time, and no candidate-specific implementation cost or duration is available here.

For a later evidence-based evaluation, the basic calculations could be:

$$
\text{Annual labor savings}
=\frac{N_{\mathrm{orders/day}}\,t_{\mathrm{saved/order}}}{60}
\,c_{\mathrm{labor/hour}}\,d_{\mathrm{operating/year}},
$$

and, for a consistently defined investment and annual benefit,

$$
\mathrm{ROI}_{1\mathrm{year}}
=100\,\frac{B_{\mathrm{annual}}-C_{\mathrm{investment}}}
{C_{\mathrm{investment}}}.
$$

Here \(t_{\mathrm{saved/order}}\) must come from observed or validated pick-time measurements, \(c_{\mathrm{labor/hour}}\) from an appropriate labor-cost definition, and \(B_{\mathrm{annual}}\) and \(C_{\mathrm{investment}}\) from documented financial estimates. These equations define what evidence would be needed; this notebook does not assign missing values or report an ROI.

## Why use a human–AI partnership?

The partnership framing is useful when a quantitative model can compare alternatives but cannot represent every implementation consideration. Computation applies the same distance calculation consistently across candidates. Human review can bring operational context, safety requirements, infrastructure knowledge, and organizational constraints into the decision—but only when qualified reviewers provide that evidence. The deterministic rubrics here demonstrate where those roles enter the workflow; they do not reproduce the judgment or expertise of actual staff.

### Strengths and limitations

- **Structured comparison:** Each candidate is evaluated using the same directed-flow and Manhattan-distance calculation.
- **Visible trade-offs:** The consensus score separates the AI alignment component from the rubric-based review component.
- **Explicit evidence gaps:** Safety, space, cost, schedule, and readiness are not fabricated from unrelated inputs.
- **Model dependence:** A low flow-distance score is useful only to the extent that the grid, flows, and distance measure represent the operation of interest.
- **Rubric dependence:** The 40%/60% weighting and review rules are illustrative choices; different documented priorities could change the ranking.
- **Implementation responsibility:** A ranked layout is not an approved design. Practical use requires operational validation, safety review, and relevant engineering and financial analysis.

### Appropriate use

This workflow can support early-stage discussion when stakeholders need a traceable comparison of candidate layouts. It is most useful when the candidate set and flow assumptions are clear and when expert review is used to add evidence rather than to lend false certainty to an incomplete model. It should not be presented as proof of realized performance, staff acceptance, safety, or financial return.

## Key results summary

Running the notebook produces a ranking for the six specified candidate layouts under the stated model and review rubrics. The ranking and plotted values are generated from the configuration when the cells run; they are not pre-recorded implementation results.

Interpret the outputs as follows:

- Flow-weighted distance is a relative model score calculated from the supplied directed flows and grid coordinates.
- Flow improvement compares a candidate with the configured current layout; it may be positive, zero, or negative.
- Pick-time reduction is a proxy obtained by multiplying flow improvement by the stated conversion factor. It is not a measured pick-time change.
- AI alignment reflects only the assessable pick-time proxy and its target. It is not a comprehensive confidence estimate across all strategic objectives.
- The human component is a deterministic, role-specific rubric score, not a survey or real expert decision.
- The selected candidate is the highest-scoring member of the refined top-three set under the stated consensus formula. Ties are reported within a numerical tolerance.
- Implementation outcomes and ROI remain undetermined until suitable measurements and cost data are supplied.

The educational contribution is the traceable sequence from model inputs to candidate comparison, review, consensus, refinement, and interpretation—together with a clear account of what the model does not establish.